# Oro: prioridades, afectaciones y trazabilidad

Consume exclusivamente las tablas plata locales. Los resultados son del inventario analizado de Cali; no representan toda la ciudad. Los datos satelitales y modelos sirven para priorizar, no para certificar seguridad ni predecir colapsos. La ocupación GEM puede repetirse entre predios; el inventario de candidatos únicos tampoco acredita personas afectadas.

Este notebook se ejecuta en `analisis_gold/ejecutados/`, carpeta privada. Contrato: `docs/ORO.md`.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
RAIZ = Path.cwd()
if not (RAIZ / 'scripts/correr_oro.py').exists():
    RAIZ = next(p for p in Path.cwd().parents if (p / 'scripts/correr_oro.py').exists())
sys.path.insert(0, str(RAIZ / 'scripts'))
from correr_oro import ejecutar_oro_local, cargar_api
tablas, reporte, destino = ejecutar_oro_local()
api = cargar_api()
plata = api['leer_plata_oro'](RAIZ / 'data/oficiales/plata')
p = tablas['predios']
plt.rcParams.update({'figure.figsize': (10, 5), 'font.size': 10})
display(pd.DataFrame({'tabla': reporte['tablas'].keys(), 'filas': reporte['tablas'].values()}))
print('Versión:', reporte['version'], '| predios:', len(p), '| en ranking:', int(p.en_ranking.sum()))

## Inventario y colas de trabajo
Los lotes se conservan sin pisos ni período. Los registros sin información satelital tienen una cola distinta. Las banderas pueden coexistir; no se deben sumar como poblaciones disjuntas.

In [ ]:
display(p.groupby(['estado_predial', 'cola_trabajo']).size().rename('n_predios').reset_index())
conteos = p.prioridad.value_counts().sort_values()
ax = conteos.plot.barh(color='#176c7e')
ax.set_xlabel('Registros prediales'); ax.set_ylabel(''); ax.set_title('Prioridad y categorías de verificación')
for i, v in enumerate(conteos): ax.text(v + 15, i, str(v), va='center')
ax.set_xlim(0, conteos.max() * 1.15)
plt.tight_layout(); plt.show()
display(tablas['verificaciones'].head(15))

## Afectaciones y denominadores
Los porcentajes por estado usan todos los predios del ámbito. La tasa de daño por comuna usa únicamente registros con dato satelital. Se informa área sumada y unión geométrica; la unión por categoría no es aditiva cuando hay solapamientos.

In [ ]:
display(tablas['afectaciones'].query("ambito == 'inventario'"))
c = tablas['comunas'].sort_values('ranking')
display(c[['comuna','n_predios','n_lotes','n_con_dato_satelital','n_con_pisos_observados','n_pisos_desconocidos_originales','pisos_prom_observados','n_en_ranking','pct_danio_entre_con_dato','indice_p90','ranking']])
ax = c.set_index('comuna')[['n_prioridad_muy_alta','n_prioridad_alta']].plot.bar(stacked=True, color=['#a42e38','#db8c37'])
ax.set_xlabel('Comuna'); ax.set_ylabel('Registros elegibles'); ax.set_title('Carga de inspección por comuna, ordenada por p90')
ax.legend(['Muy alta','Alta']); plt.tight_layout(); plt.show()

## Por qué aparece primero cada predio
Se preservan pesos base y normalización. Los pesos efectivos se redistribuyen entre componentes disponibles. Las contribuciones suman el índice antes de redondear. Este puntaje es relativo al inventario, no una probabilidad.

In [ ]:
top = p[p.en_ranking].sort_values('posicion').head(10)
display(top[['id','comuna','posicion','prioridad','indice_riesgo','motivos_principales','n_componentes_disponibles','peso_base_disponible','gem_ambiguo','gem_reutilizado']])
aportes = top.set_index('id')[[f'aporte_{k}' for k in api['PESOS']]]
aportes.columns = [k.removeprefix('aporte_c_') for k in aportes.columns]
ax = aportes.plot.barh(stacked=True, colormap='tab20')
ax.invert_yaxis(); ax.set_xlabel('Contribución al puntaje'); ax.set_ylabel('ID del predio')
ax.set_title('Componentes del top 10'); ax.legend(loc='upper left', bbox_to_anchor=(1,1))
plt.tight_layout(); plt.show()

## Exposición y servicios
La primera suma puede repetir registros GEM entre predios. La segunda cuenta una vez cada candidato GEM vinculado, pero sigue siendo un modelo y no un censo de afectados. Los servicios cercanos no acreditan capacidad, operatividad ni tiempo de viaje.

In [ ]:
ex = tablas['exposicion_gem']
display(pd.DataFrame({'indicador':['Ocupación GEM asignada a predios (puede repetirse)','Ocupación de candidatos GEM contados una vez','Candidatos GEM únicos vinculados','Predios con candidato reutilizado'],
                      'valor':[p.ocupantes_asignados_gem.sum(min_count=1), ex.ocupantes_noche.sum(min_count=1), len(ex), int(p.gem_reutilizado.sum())]}))
display(ex.groupby('comuna_gem',dropna=False).agg(n_registros_gem=('id','size'),ocupacion_modelada=('ocupantes_noche',lambda s:s.sum(min_count=1))).reset_index())
display(tablas['servicios'].groupby(['fuente','urgencias'],dropna=False).agg(n_servicios=('id','size'),con_capacidad_informada=('capacidad','count'),con_operatividad_informada=('operatividad','count')).reset_index())

## Sensibilidad y factores asociados
Se conserva el escenario anterior bajo `sin_vulnerabilidad_ni_suelo`; `sin_fuentes_gem` ahora excluye también exposición. Las pruebas de factores son exploratorias, con control de frecuencias esperadas y ajuste Holm para las admisibles. Un p-valor alto no prueba ausencia de relación; no se acredita independencia espacial ni causalidad.

In [ ]:
display(tablas['resumen_sensibilidad'])
r = tablas['resumen_sensibilidad']
ax = r.set_index('escenario').pct_top_coincide.plot.barh(color='#176c7e')
ax.set_xlim(0,105); ax.set_xlabel('% de coincidencia con el top base'); ax.set_ylabel('')
ax.set_title('Estabilidad del top de inspección'); plt.tight_layout(); plt.show()
display(tablas['factores'])
display(tablas['grupos_factores'])

## Controles y reproducción
Los controles se ejecutan antes de escribir las tablas. Los hashes físicos corresponden a las entradas Parquet de plata de esta corrida. La comparación entre ciudades y la publicación de estas tablas en Databricks siguen pendientes.

In [ ]:
controles = api['validar_oro'](tablas, plata)
assert controles.cumple.all()
assert len(p) == len(plata['predios'])
assert not p.loc[p.sin_construccion_registrada, 'en_ranking'].any()
assert p.loc[p.sin_construccion_registrada, ['pisos','pisos_estimados','periodo_s']].isna().all().all()
display(controles)
display(tablas['parametros'])
display(tablas['fuentes'][['fuente','tabla_plata','corte_estado']])
print(json.dumps(reporte, ensure_ascii=False, indent=2))
print('Verificaciones completas. Resultados guardados en:', destino)